In [ ]:
def load_and_prepare_datasets(depth, path="saved_files/dataset"):
    """
    Carga y prepara dataframes para una profundidad específica.
    
    Args:
        depth: Profundidad (ej: "in_0_1", "in_1_2")
        datasets_to_keep: Lista de nombres de datasets a mantener (opcional) - Puesto a mano dentro de la función
        path: Ruta al directorio con los archivos CSV
    
    Returns:
        dict: Diccionario {nombre_dataset: dataframe_preparado}
    """
    # Cargamos los csv de los tifs
    dfs = {}
    for archivo in os.listdir(path):
        if archivo.endswith(f"{depth}_features.csv"):
            nombre_sin_extension = os.path.splitext(archivo)[0]  # sin .csv
            ruta_completa = os.path.join(path, archivo)
            dfs[nombre_sin_extension[:-9]] = pd.read_csv(ruta_completa)
    
    # Limpiamos valores nulos
    for nombre_df, df in dfs.items():
        for band_set in ["rhow", "rhown", "rtoa"]:
            dfs[nombre_df] = df.dropna()
    
    # Filtrar datasets si se especifica

    datasets_to_keep = [
        'C2X-Complex_rhow_15x15_depth_in_0_1', 
        'C2X-Complex_rhow_15x15_depth_in_1_2',
        'TOA_15x15_depth_in_2_3',
        'TOA_15x15_depth_in_3_4'
        ]

    if datasets_to_keep is not None:
        dfs = {k: dfs[k] for k in datasets_to_keep if k in dfs}
    
    # Procesamiento de cada dataframe
    for nombre_df, df in dfs.items():
        # Marcamos las columnas de Chl alta (equivalente a quantile(0.9))
        threshold = df["Chl"].quantile(0.9)
        df["High_Chl"] = df["Chl"] > threshold
        
        # Sacamos la estación de cada fecha
        df['Date'] = pd.to_datetime(df['Date'])
        
        def get_season(month):
            if month in [12, 1, 2]:
                return 'Invierno'
            elif month in [3, 4, 5]:
                return 'Primavera'
            elif month in [6, 7, 8]:
                return 'Verano'
            else:
                return 'Otoño'
        
        df['Season'] = df['Date'].dt.month.apply(get_season)
        
        # Convertir a categorías y hacer one-hot encoding
        for col in df.select_dtypes(include=['object', 'string']).columns:
            df[col] = df[col].astype('category')
        
        df = pd.concat([df.drop(columns=["Season"]), pd.get_dummies(df["Season"])], axis=1)
        dfs[nombre_df] = df
    
    return dfs

In [ ]:
dfs = load_and_prepare_datasets("") # argumento de depth vacío para que los coja todos

In [ ]:
"""
=============================================================================
MODELOS FINALES - Entrenamiento con Optuna sobre dataset completo
=============================================================================

Flujo:
  1. Optuna busca los mejores hiperparámetros de CAT usando validación cruzada
     sobre el dataset COMPLETO (sin split train/test).
  2. Con los mejores parámetros encontrados, se entrena el modelo final
     sobre todos los datos.
  3. Se guarda el modelo, las features y los metadatos.

Datasets seleccionados (uno por profundidad):
  - in_0_1 -> C2X-Complex_rhow_15x15_depth_in_0_1 + CAT
  - in_1_2 -> C2X-Complex_rhow_15x15_depth_in_1_2 + CAT
  - in_2_3 -> TOA_15x15_depth_in_2_3              + CAT
  - in_3_4 -> TOA_15x15_depth_in_3_4              + CAT
=============================================================================
"""

import os
import json
import platform
import numpy as np
import pandas as pd
import optuna
import joblib
import pickle

from catboost import CatBoostRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import r2_score, mean_squared_log_error

optuna.logging.set_verbosity(optuna.logging.WARNING)

# ─────────────────────────────────────────────
# CONFIGURACIÓN
# ─────────────────────────────────────────────
TARGET        = "Chl"
FOLDS         = 5          # Número de folds para CV
N_TRIALS      = 100        # Trials de Optuna
SEED          = 42
SAVE_PATH     = "training_results/models"
HYPERPARAMS   = "hyperparams.json"

os.makedirs(SAVE_PATH, exist_ok=True)

# Datasets seleccionados (resultado del análisis multi-seed)
SELECTED = {
    "C2X-Complex_rhow_15x15_depth_in_0_1": dfs["C2X-Complex_rhow_15x15_depth_in_0_1"],
    "C2X-Complex_rhow_15x15_depth_in_1_2": dfs["C2X-Complex_rhow_15x15_depth_in_1_2"],
    "TOA_15x15_depth_in_2_3":              dfs["TOA_15x15_depth_in_2_3"],
    "TOA_15x15_depth_in_3_4":              dfs["TOA_15x15_depth_in_3_4"],
}

# ─────────────────────────────────────────────
# FUNCIÓN OBJETIVO OPTUNA (CV sobre dataset completo)
# ─────────────────────────────────────────────
def objective_cat(trial, df, seed):
    """
    Función objetivo para CAT.
    Usa validación cruzada estratificada sobre el dataset completo.
    Retorna (rmsle_mean, r2_mean) para optimización multi-objetivo.
    """
    # Cargar espacio de búsqueda desde hyperparams.json
    with open(HYPERPARAMS, "r") as f:
        config = json.load(f)["CAT"]
    param_config = config["params"]

    # Construir parámetros del trial
    params = {}
    for param_name, param_spec in param_config.items():
        if isinstance(param_spec, dict) and "type" in param_spec:
            ptype = param_spec["type"]
            if ptype == "categorical":
                params[param_name] = trial.suggest_categorical(param_name, param_spec["choices"])
            elif ptype == "int":
                params[param_name] = trial.suggest_int(param_name, param_spec["low"], param_spec["high"])
            elif ptype == "float":
                params[param_name] = trial.suggest_float(
                    param_name, param_spec["low"], param_spec["high"],
                    log=param_spec.get("log", False)
                )
        else:
            # Parámetro fijo
            params[param_name] = param_spec

    # Preparar X e y (ignorar Date, Lat, Lon, Buoy)
    df_work = df.iloc[:, 4:].copy()
    X = df_work.drop(columns=[TARGET, "High_Chl", "Turbidez"])
    y = df_work[TARGET]
    y_class = df_work["High_Chl"]   # Para StratifiedKFold

    rmsle_folds, r2_folds = [], []
    skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=seed)

    for train_idx, val_idx in skf.split(X, y_class):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model = CatBoostRegressor(**params)
        model.fit(X_train, y_train, eval_set=(X_val, y_val), verbose=False)

        val_pred = model.predict(X_val)
        val_pred = np.clip(val_pred, 0.2, None)   # Forzar predicciones > 0

        rmsle_folds.append(np.sqrt(mean_squared_log_error(y_val, val_pred)))
        r2_folds.append(r2_score(y_val, val_pred))

    return np.mean(rmsle_folds), np.mean(r2_folds)


# ─────────────────────────────────────────────
# FUNCIÓN PRINCIPAL
# ─────────────────────────────────────────────
def run_final_cat(selected_dfs, n_trials=N_TRIALS, seed=SEED):
    """
    Para cada dataset seleccionado:
      1. Ejecuta Optuna (multi-objetivo: minimizar RMSLE, maximizar R2)
         sobre el dataset completo con CV.
      2. Selecciona el mejor trial del frente de Pareto.
      3. Entrena el modelo final con todos los datos.
      4. Guarda modelo, features y metadatos.
    """
    all_results = {}

    for nombre_df, df in selected_dfs.items():
        print(f"\n{'='*65}")
        print(f"Optimizando CAT para: {nombre_df}")
        print(f"   Dataset: {len(df)} muestras | {df.shape[1]} columnas")
        print(f"   Trials Optuna: {n_trials} | Folds CV: {FOLDS}")
        print(f"{'='*65}")

        # ── 1. Optuna multi-objetivo ──────────────────────────────────
        study = optuna.create_study(
            directions=["minimize", "maximize"],   # RMSLE min, R2 max
            sampler=optuna.samplers.TPESampler(
                constant_liar=True,
                n_startup_trials=10,
                seed=seed
            )
        )
        study.optimize(
            lambda trial: objective_cat(trial, df, seed),
            n_trials=n_trials,
            n_jobs=1,          # CatBoost ya usa múltiples threads internamente
            show_progress_bar=True
        )

        # ── 2. Selección del mejor trial (frente de Pareto) ──────────
        pareto_trials = study.best_trials
        if not pareto_trials:
            pareto_trials = study.trials

        # Criterio de selección: score combinado
        # 50% RMSLE normalizado + 35% R2 normalizado (invertido) + 15% estabilidad
        rmsle_vals = np.array([t.values[0] for t in pareto_trials])
        r2_vals    = np.array([t.values[1] for t in pareto_trials])

        if len(pareto_trials) > 1:
            rmsle_norm = (rmsle_vals - rmsle_vals.min()) / (rmsle_vals.max() - rmsle_vals.min() + 1e-9)
            r2_norm    = 1 - (r2_vals - r2_vals.min()) / (r2_vals.max() - r2_vals.min() + 1e-9)
            score      = 0.65 * rmsle_norm + 0.35 * r2_norm
            best_trial = pareto_trials[int(np.argmin(score))]
        else:
            best_trial = pareto_trials[0]

        best_params  = best_trial.params
        best_rmsle   = best_trial.values[0]
        best_r2      = best_trial.values[1]

        print(f"\nMejor trial encontrado:")
        print(f"   RMSLE CV: {best_rmsle:.4f}")
        print(f"   R2 CV:    {best_r2:.4f}")
        print(f"   Params:   {best_params}")

        # ── 3. Parámetros finales: fijos del JSON + optimizados ───────
        with open(HYPERPARAMS, "r") as f:
            config = json.load(f)["CAT"]

        final_params = {}
        for param_name, param_spec in config["params"].items():
            if not isinstance(param_spec, dict) or "type" not in param_spec:
                final_params[param_name] = param_spec   # parámetros fijos
        final_params.update(best_params)                # sobrescribir con optimizados

        # ── 4. Entrenamiento final sobre TODOS los datos ──────────────
        df_work = df.iloc[:, 4:].copy()
        X_full  = df_work.drop(columns=[TARGET, "High_Chl", "Turbidez"])
        y_full  = df_work[TARGET]
        feature_names = X_full.columns.tolist()

        print(f"\nEntrenando modelo final sobre {len(X_full)} muestras...")
        final_model = CatBoostRegressor(**final_params)
        final_model.fit(X_full, y_full, verbose=False)
        print(f"   Modelo entrenado")

        # ── 5. Guardar artefactos ─────────────────────────────────────
        model_name = "CAT"
        base_name  = f"{nombre_df}_{model_name}"

        # 5a. Modelo joblib
        joblib_path = os.path.join(SAVE_PATH, f"{base_name}_model.joblib")
        joblib.dump(final_model, joblib_path)

        # 5b. Modelo nativo CatBoost (.cbm)
        cbm_path = os.path.join(SAVE_PATH, f"{base_name}_model.cbm")
        final_model.save_model(cbm_path)

        # 5c. Features
        features_path = os.path.join(SAVE_PATH, f"{base_name}_features.json")
        with open(features_path, "w") as f:
            json.dump({"feature_names": feature_names}, f, indent=2)

        # 5d. Metadatos
        import catboost, sklearn, numpy, pandas as pd_meta
        meta = {
            "dataset":      nombre_df,
            "model_name":   model_name,
            "best_params":  final_params,
            "cv_rmsle":     round(best_rmsle, 4),
            "cv_r2":        round(best_r2, 4),
            "n_trials":     n_trials,
            "n_folds":      FOLDS,
            "n_samples":    len(X_full),
            "n_features":   len(feature_names),
            "python":       platform.python_version(),
            "libs": {
                "catboost": catboost.__version__,
                "sklearn":  sklearn.__version__,
                "numpy":    numpy.__version__,
                "pandas":   pd_meta.__version__,
            }
        }
        meta_path = os.path.join(SAVE_PATH, f"{base_name}_metadata.json")
        with open(meta_path, "w") as f:
            json.dump(meta, f, indent=2)

        print(f"   Guardado en: {SAVE_PATH}/")
        print(f"      - {base_name}_model.joblib")
        print(f"      - {base_name}_model.cbm")
        print(f"      - {base_name}_features.json")
        print(f"      - {base_name}_metadata.json")

        all_results[nombre_df] = {
            "model":       final_model,
            "best_params": final_params,
            "cv_rmsle":    best_rmsle,
            "cv_r2":       best_r2,
        }

    print(f"\n{'='*65}")
    print("ENTRENAMIENTO FINAL COMPLETADO")
    print(f"{'='*65}\n")

    # Resumen
    print(f"{'Dataset':<45} {'RMSLE CV':>10} {'R2 CV':>8}")
    print("-" * 65)
    for nombre_df, res in all_results.items():
        print(f"{nombre_df:<45} {res['cv_rmsle']:>10.4f} {res['cv_r2']:>8.4f}")

    return all_results


In [ ]:
# ─────────────────────────────────────────────
# EJECUCIÓN
# ─────────────────────────────────────────────
final_results = run_final_cat(SELECTED, n_trials=N_TRIALS, seed=SEED)